# Global earthquake clustering — Phases 1-4 + Bonus
This notebook only clones/pulls the GitHub repo and runs the logic from `src/`.
To change the code, edit the files under `src/` on GitHub and re-run this notebook (the clone cell pulls automatically).


Computational Intelligence Course Project — Ferdowsi University of Mashhad, Department of Computer Engineering

Team members' names and student IDs: Amirhossein Kooshesh 4011262429

This notebook reproduces Phases 1-4 (plus the optional bonus section) of the project: Phase 1 cleans the global earthquake dataset, visualizes event locations and selected regions, computes local density via nearest-neighbor distances, and records predictions P1-P4 before any clustering is run. Phase 2 runs K-Means (raw lat/lon vs. the 3D unit-sphere representation, choice of k, and cluster analysis). Phase 3 runs DBSCAN on the full dataset and Single/Average Link hierarchical clustering on the shared sample S. Phase 4 evaluates all four methods together on S (summary table, pairwise ARI, resampling-stability experiment) and concludes with the P1-P4 verdicts and the final per-scenario recommendation. The bonus section (optional) covers a controlled-perturbation robustness check for K-Means/DBSCAN and an HDBSCAN-vs-DBSCAN comparison. All logic lives in `src/`; this notebook only clones the repo and calls it.


In [ ]:
import os

CONTENT_DIR = "/content"  # fixed, known root -- re-running this cell is always safe
REPO_URL = "https://github.com/AHK-0214/CI_Clustering.git"
REPO_DIR = os.path.join(CONTENT_DIR, "CI_Clustering")

os.chdir(CONTENT_DIR)  # undo any previous %cd before deciding anything

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print("Repo already exists; pulling latest changes...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("Current directory:", os.getcwd())

# Force a fresh import of src/* in the cell below, even if this kernel already
# imported an older version of it earlier in this session. Without this, a
# `git pull` updates the files on disk but Python keeps serving the OLD cached
# module from sys.modules, which is the classic cause of a stale
# `KeyError: 'p1_structures'` (or any other "it's right there in the code but
# still fails") after pulling new code without restarting the runtime.
import sys
for _mod_name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_mod_name]


In [ ]:
!pip install -q -r requirements.txt

## Dataset
Pick one of two options:

**Option 1 (recommended, simplest):** download `database.csv` from Kaggle and upload it here.

**Option 2:** upload your own `kaggle.json` so kagglehub can download it automatically.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

USE_KAGGLEHUB = False  # set to True to try automatic download via kagglehub

if USE_KAGGLEHUB:
    from google.colab import files
    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        os.rename(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
else:
    if not os.path.exists("data/database.csv"):
        from google.colab import files
        print("Upload the database.csv file (from the usgs/earthquake-database Kaggle dataset):")
        uploaded = files.upload()
        for fname in uploaded:
            os.rename(fname, "data/database.csv")
    else:
        print("data/database.csv already exists.")

In [ ]:
import sys, importlib
sys.path.insert(0, ".")

# Import (or re-import) the current on-disk version of src.phase1 -- see the
# cache-busting note in the clone/pull cell above.
import src.phase1 as _phase1_mod
importlib.reload(_phase1_mod)
from src.phase1 import run_phase1

import matplotlib.pyplot as plt
plt.ioff()  # run_phase1() builds several matplotlib figures; ioff() plus the
            # plt.close(fig) calls inside src/viz.py stop them from being
            # auto-printed here a second time -- each figure is only shown once,
            # later, in its own cell under Questions A/B/D below.
results = run_phase1(save_outputs=True)
plt.ion()

print("Total earthquakes after cleaning (Type == Earthquake):", results["n_total"])
print("Base seed (s):", results["seed_base"])


## Question A — map and three regions

In [ ]:
results["figures"]["map"]

Real-world map with country borders/coastlines (hover to see lat/lon), so each earthquake can be tied to an actual country or region instead of a bare scatter plot:

In [ ]:
results["figures"]["map_geo"]

**Note:** the region near ±180° longitude (e.g. the Tonga-Kermadec trench) may look split into two pieces at the left/right edge of the flat map above, because any flat map of a sphere necessarily has a "seam" somewhere. This is only a 2D-display limitation; the distance calculations (conversion to xyz on the unit sphere in `src/geo.py`) do not have this problem, and points on either side of this line are correctly treated as near each other. (Full explanation: Section C of the English report, `report_phase1_en.docx`.)


In [ ]:
results["figures"]["map_regions"]

In [ ]:
results["region_table"]

## Question B — density (distance to the 10th neighbor)

In [ ]:
results["figures"]["hist"]

In [ ]:
results["density_table"]

## Question C — why plain Euclidean distance on lat/lon is misleading

In [ ]:
print(results["distance_explanation"])
print()
print("(Also written out in full in Section C of the English report, report_phase1_en.docx.)")


## Question D — predictions P1 through P4

In [ ]:
print("Predictions P1-P4 (prediction, reason, rejection criterion) are also written out "
      "in full in Section D of the English report (report_phase1_en.docx). The table below "
      "shows the same predictions dict used to generate that section.")
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.DataFrame(results["predictions"]).T


### Question D (continued) — where are the structures named in P1?
This is a subsection of Question D / prediction P1 above, not a separate question. Still raw data (no clustering algorithm has been run) — just locating the candidate elongated structures mentioned in P1's reasoning (Mid-Atlantic Ridge, South America's west coast, the East Asia / western Pacific arc) on the map.


In [ ]:
results["figures"]["p1_structures"]


# Phase 2 — K-Means


In [ ]:
import src.phase2 as _phase2_mod
importlib.reload(_phase2_mod)
from src.phase2 import run_phase2

results2 = run_phase2(df=results["df"])
print(f"Selected k = {results2['final_k']}")


## Question A — raw (lat, lon) vs. 3D sphere, and the initialization baseline


In [ ]:
results2["figures"]["kmeans_2d"]


In [ ]:
results2["figures"]["kmeans_3d"]


In [ ]:
print(f"ARI (raw lat/lon vs 3D sphere, both k={results2['final_k']}, seed=s): "
      f"{results2['ari_2d_vs_3d']:.3f}")
print(f"ARI (3D sphere, seed=s vs seed=s+1, pure init-variability baseline): "
      f"{results2['ari_3d_baseline_init']:.3f}")
print()
print(results2["qa_discussion"])


## Question B — choosing k


In [ ]:
results2["figures"]["inertia"]


In [ ]:
results2["kb_table"]  # inertia + Silhouette (computed only on sample S) for k in {8,12,16,20}


In [ ]:
print(results2["selected_k_justification"])


## Question C — analysis of the selected k output (revisiting P1)


In [ ]:
results2["figures"]["kmeans_selected"]


In [ ]:
print("Cluster breakdown per Phase-1 region (cluster_id, count), largest first:")
for region, breakdown in results2["region_cluster_breakdown"].items():
    print(f"  {region}: {breakdown}")
print()
print(results2["qc_discussion"])


## Question D — geographic spread (max pairwise distance) per cluster


In [ ]:
results2["span_table"]  # max_span_km per cluster; is_lower_bound_estimate flags subsampled (>1500-pt) clusters


In [ ]:
print(results2["qd_discussion"])


# Phase 3 — DBSCAN and hierarchical clustering


In [ ]:
import src.phase3 as _phase3_mod
importlib.reload(_phase3_mod)
from src.phase3 import run_phase3

results3 = run_phase3(df=results["df"], sample_S=results2["sample_S"])
print("DBSCAN eps values plotted below:", results3["qa_eps_plotted"])


## Question A — DBSCAN on the full data (eps in {75, 200, 500} km)


In [ ]:
results3["qa_table"]  # n_clusters, noise%, largest cluster size/share, # clusters with 10-19 points


In [ ]:
results3["figures"]["dbscan"][results3["qa_eps_plotted"][0]]


In [ ]:
results3["figures"]["dbscan"][results3["qa_eps_plotted"][1]]


In [ ]:
print(results3["qa_discussion"])


## Question B — density per region, per eps


In [ ]:
results3["qb_table"]  # per (eps, region): % noise, % in main cluster, shared main cluster with another region


In [ ]:
print(results3["qb_conclusion"])
print()
print(f"Final eps chosen for scenario B: {results3['final_eps_scenario_b']} km")
print(results3["qb_final_eps_reason"])


## Question C — hierarchical clustering on the shared sample S


In [ ]:
results3["figures"]["single_link"]


In [ ]:
results3["figures"]["average_link"]


In [ ]:
print("Single Link cluster sizes:"); display(results3["size_single"])
print("Average Link cluster sizes:"); display(results3["size_average"])


In [ ]:
print(results3["qc_discussion"])


## Question D — the chaining effect


In [ ]:
results3["figures"]["chain"]


In [ ]:
print(results3["qd_discussion"])


# Phase 4 — Evaluation and conclusion


In [ ]:
import src.phase4 as _phase4_mod
importlib.reload(_phase4_mod)
from src.phase4 import run_phase4

results4 = run_phase4(df=results["df"], sample_S=results2["sample_S"],
                       phase2_results=results2, phase3_results=results3)
print("DBSCAN eps used for the 4-method comparison (selected for scenario B in Phase 3-B):",
      results3["final_eps_scenario_b"], "km")


## Question A — evaluation on the shared sample S
Four label sets on the same 2000 points of S: K-Means and DBSCAN labels extracted (by `record_id`) from the full-data runs selected in Phases 2-3, next to Single Link and Average Link, which were fit on S directly.


In [ ]:
results4["qa_summary_table"]  # n_clusters, largest/median size, # small clusters, coverage%, Silhouette


Same-extent maps, all four methods:


In [ ]:
for name, fig in results4["figures"].items():
    display(fig)


Pairwise ARI (all 6 pairs; DBSCAN's noise label is kept as its own group):


In [ ]:
results4["qa_ari_table"]


In [ ]:
results4["qa_ari_matrix"]


In [ ]:
print(results4["qa_eval_discussion"])


### Resampling stability
Three 80%-of-S subsets (seeds s, s+1, s+2); K-Means (k=12), Single Link and Average Link run on each (9 runs total); pairwise ARI on each pair's common points (9 ARI values).


In [ ]:
results4["stability_pairs_table"]  # 9 rows: method x subset-pair


In [ ]:
results4["stability_summary_table"]  # mean/range of ARI per method


In [ ]:
print(results4["stability_discussion"])


In [ ]:
print(results4["best_numeric_but_not_best_fit"])


## Question B — conclusion


In [ ]:
results4["verdict_table"]  # P1-P4: confirmed / rejected / partially confirmed, with evidence


In [ ]:
import json
print(json.dumps(results4["final_recommendation"], indent=2))
print()
print(results4["same_algorithm_note"])


# Bonus Q.A — Controlled Perturbation (optional, +5)
5% of the full dataset, displaced 10-20 km in a random geographic direction (seed s+3); K-Means and DBSCAN re-run with the project's selected parameters, original vs. perturbed.


In [ ]:
import src.bonus as _bonus_mod
importlib.reload(_bonus_mod)
from src.bonus import run_bonus_qa, run_bonus_qb

bonus_qa = run_bonus_qa(df=results["df"], phase2_results=results2, phase3_results=results3)
bonus_qa["qa_table"]


In [ ]:
print(bonus_qa["qa_discussion"])


In [ ]:
if bonus_qa["figures"]["example"] is not None:
    display(bonus_qa["figures"]["example"])


# Bonus Q.B — HDBSCAN (optional, +5)
HDBSCAN vs. DBSCAN on the shared sample S, with DBSCAN's eps re-tuned on S itself.


In [ ]:
bonus_qb = run_bonus_qb(df=results["df"], sample_S=results2["sample_S"])
print(bonus_qb["hdb_explanation"])


In [ ]:
bonus_qb["candidates_table"]  # DBSCAN eps candidates tried on S


In [ ]:
print(bonus_qb["eps_choice_reason"])


In [ ]:
bonus_qb["overall_table"]


In [ ]:
display(bonus_qb["figures"]["dbscan"])
display(bonus_qb["figures"]["hdbscan"])


In [ ]:
bonus_qb["region_table"]  # per Phase-1 region: n in S, DBSCAN noise%, HDBSCAN noise%


In [ ]:
print(bonus_qb["verdict_discussion"])


## Download outputs (to attach to the report)
All files are saved under `outputs/phase1/`, `outputs/phase2/`, `outputs/phase3/`, `outputs/phase4/` and `outputs/bonus/`; the cell below zips and downloads all of them together.


In [ ]:
!zip -r all_phases_outputs.zip outputs/phase1 outputs/phase2 outputs/phase3 outputs/phase4 outputs/bonus
from google.colab import files
files.download("all_phases_outputs.zip")
